In [3]:
import pandas as pd

df = pd.read_csv("dSets/synthetic_orders_model.csv")

df["has_return_history"] = df["customer_past_return_rate"].notna().astype(int)

mean_return_rate = df["customer_past_return_rate"].mean()
df["customer_past_return_rate"] = df["customer_past_return_rate"].fillna(mean_return_rate)

print(f"Population mean return rate used for imputation: {mean_return_rate:.4f}")
print(f"Orders with known history: {df['has_return_history'].sum()} / {len(df)}")

categorical_cols = ["category", "payment_method", "delivery_pincode_tier", "time_of_day_ordered"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

id_cols = ["order_id", "customer_id", "order_date"]
df_encoded = df_encoded.drop(columns=id_cols)

print(f"\nShape before encoding: {df.shape}")
print(f"Shape after encoding: {df_encoded.shape}")
print(f"\nColumns after encoding:\n{list(df_encoded.columns)}")

df_encoded.to_csv("dSets/synthetic_orders_prepped.csv", index=False)
print("\nSaved: dSets/synthetic_orders_prepped.csv")

Population mean return rate used for imputation: 0.2500
Orders with known history: 4338 / 5000

Shape before encoding: (5000, 16)
Shape after encoding: (5000, 25)

Columns after encoding:
['order_value', 'discount_pct', 'is_first_time_buyer', 'customer_past_orders', 'customer_past_return_rate', 'size_variant_flag', 'days_to_deliver', 'returned', 'has_return_history', 'category_beauty', 'category_electronics', 'category_fashion', 'category_grocery', 'category_home', 'payment_method_COD', 'payment_method_UPI', 'payment_method_card', 'payment_method_netbanking', 'delivery_pincode_tier_metro', 'delivery_pincode_tier_tier2', 'delivery_pincode_tier_tier3', 'time_of_day_ordered_afternoon', 'time_of_day_ordered_evening', 'time_of_day_ordered_morning', 'time_of_day_ordered_night']

Saved: dSets/synthetic_orders_prepped.csv


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_csv("dSets/synthetic_orders_prepped.csv")

X = df.drop(columns=["returned"])
y = df["returned"]

X_trainpool, X_test, y_trainpool, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training pool: {X_trainpool.shape[0]} orders, return rate: {y_trainpool.mean():.3f}")
print(f"Held-out test: {X_test.shape[0]} orders, return rate: {y_test.mean():.3f}")

X_test.to_csv("dSets/X_test_FINAL.csv", index=False)
y_test.to_csv("dSets/y_test_FINAL.csv", index=False)


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

gb_model = GradientBoostingClassifier(random_state=42)

scoring = ["roc_auc", "average_precision"]  # average_precision = PR-AUC

print("\n--- Logistic Regression (5-fold CV) ---")
log_reg_scores = cross_validate(log_reg_pipeline, X_trainpool, y_trainpool, cv=cv, scoring=scoring)
print(f"ROC-AUC: {log_reg_scores['test_roc_auc'].mean():.4f} ± {log_reg_scores['test_roc_auc'].std():.4f}")
print(f"PR-AUC:  {log_reg_scores['test_average_precision'].mean():.4f} ± {log_reg_scores['test_average_precision'].std():.4f}")

print("\n--- Gradient Boosting (5-fold CV) ---")
gb_scores = cross_validate(gb_model, X_trainpool, y_trainpool, cv=cv, scoring=scoring)
print(f"ROC-AUC: {gb_scores['test_roc_auc'].mean():.4f} ± {gb_scores['test_roc_auc'].std():.4f}")
print(f"PR-AUC:  {gb_scores['test_average_precision'].mean():.4f} ± {gb_scores['test_average_precision'].std():.4f}")

X_trainpool.to_csv("dSets/X_trainpool.csv", index=False)
y_trainpool.to_csv("dSets/y_trainpool.csv", index=False)

Training pool: 4000 orders, return rate: 0.200
Held-out test: 1000 orders, return rate: 0.200

--- Logistic Regression (5-fold CV) ---
ROC-AUC: 0.7481 ± 0.0244
PR-AUC:  0.4463 ± 0.0378

--- Gradient Boosting (5-fold CV) ---
ROC-AUC: 0.7384 ± 0.0165
PR-AUC:  0.4350 ± 0.0316


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

X_trainpool = pd.read_csv("dSets/X_trainpool.csv")
y_trainpool = pd.read_csv("dSets/y_trainpool.csv").squeeze()
X_test = pd.read_csv("dSets/X_test_FINAL.csv")
y_test = pd.read_csv("dSets/y_test_FINAL.csv").squeeze()

log_reg_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
log_reg_final.fit(X_trainpool, y_trainpool)

gb_final = GradientBoostingClassifier(random_state=42)
gb_final.fit(X_trainpool, y_trainpool)

log_reg_test_probs = log_reg_final.predict_proba(X_test)[:, 1]
gb_test_probs = gb_final.predict_proba(X_test)[:, 1]

print("=== FINAL HELD-OUT TEST SET RESULTS (evaluated once) ===\n")

print("Logistic Regression (PRIMARY MODEL):")
print(f"  ROC-AUC: {roc_auc_score(y_test, log_reg_test_probs):.4f}")
print(f"  PR-AUC:  {average_precision_score(y_test, log_reg_test_probs):.4f}")

print("\nGradient Boosting (comparison model):")
print(f"  ROC-AUC: {roc_auc_score(y_test, gb_test_probs):.4f}")
print(f"  PR-AUC:  {average_precision_score(y_test, gb_test_probs):.4f}")

pd.DataFrame({
    "y_true": y_test,
    "log_reg_prob": log_reg_test_probs,
    "gb_prob": gb_test_probs,
}).to_csv("dSets/test_predictions.csv", index=False)

print("\nSaved: dSets/test_predictions.csv")

=== FINAL HELD-OUT TEST SET RESULTS (evaluated once) ===

Logistic Regression (PRIMARY MODEL):
  ROC-AUC: 0.7710
  PR-AUC:  0.4753

Gradient Boosting (comparison model):
  ROC-AUC: 0.7535
  PR-AUC:  0.4728

Saved: test_predictions.csv


In [11]:
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

preds = pd.read_csv("dSets/test_predictions.csv")

y_true = preds["y_true"]
log_reg_probs = preds["log_reg_prob"]

y_pred_default = (log_reg_probs >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred_default)
tn, fp, fn, tp = cm.ravel()

print("=== Confusion Matrix (threshold = 0.5) ===")
print(f"                  Predicted: Safe   Predicted: Risky")
print(f"Actually Safe:    {tn:>13}   {fp:>15}")
print(f"Actually Returned:{fn:>13}   {tp:>15}")

print(f"\nTrue Positives (caught real returns):   {tp}")
print(f"False Positives (false alarms):         {fp}")
print(f"False Negatives (missed real returns):  {fn}")
print(f"True Negatives (correctly let through):  {tn}")

print(f"\nPrecision: {precision_score(y_true, y_pred_default):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred_default):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred_default):.4f}")

print("\nFull classification report:")
print(classification_report(y_true, y_pred_default, target_names=["Not Returned", "Returned"]))

=== Confusion Matrix (threshold = 0.5) ===
                  Predicted: Safe   Predicted: Risky
Actually Safe:              781                19
Actually Returned:          167                33

True Positives (caught real returns):   33
False Positives (false alarms):         19
False Negatives (missed real returns):  167
True Negatives (correctly let through):  781

Precision: 0.6346
Recall:    0.1650
F1 Score:  0.2619

Full classification report:
              precision    recall  f1-score   support

Not Returned       0.82      0.98      0.89       800
    Returned       0.63      0.17      0.26       200

    accuracy                           0.81      1000
   macro avg       0.73      0.57      0.58      1000
weighted avg       0.79      0.81      0.77      1000



In [13]:
import pandas as pd
import numpy as np

preds = pd.read_csv("dSets/test_predictions.csv")

X_test = pd.read_csv("dSets/X_test_FINAL.csv")
preds["order_value"] = X_test["order_value"].values

y_true = preds["y_true"]
probs = preds["log_reg_prob"]
order_value = preds["order_value"]

FP_COST = 40  # flat cost per false alarm: review/friction overhead

def fn_cost(order_value):
    return 150 + 50 + 0.02 * order_value  # reverse logistics + restocking + refund fees


thresholds = np.arange(0.05, 0.95, 0.01)
results = []

for t in thresholds:
    y_pred = (probs >= t).astype(int)

    fp_mask = (y_pred == 1) & (y_true == 0)
    fn_mask = (y_pred == 0) & (y_true == 1)

    total_fp_cost = fp_mask.sum() * FP_COST
    total_fn_cost = fn_cost(order_value[fn_mask]).sum()
    total_cost = total_fp_cost + total_fn_cost

    results.append({
        "threshold": round(t, 2),
        "n_flagged": y_pred.sum(),
        "false_positives": fp_mask.sum(),
        "false_negatives": fn_mask.sum(),
        "fp_cost": total_fp_cost,
        "fn_cost": round(total_fn_cost, 2),
        "total_cost": round(total_cost, 2),
    })

results_df = pd.DataFrame(results)

best_row = results_df.loc[results_df["total_cost"].idxmin()]
default_row = results_df.loc[results_df["threshold"] == 0.5]

print("=== Cost at default threshold (0.5) ===")
print(default_row.to_string(index=False))

print("\n=== Cost-optimal threshold ===")
print(best_row.to_string(index=False))

print(f"\nSavings from switching to optimal threshold: "
      f"₹{default_row['total_cost'].values[0] - best_row['total_cost']:.2f}")

results_df.to_csv("dSets/threshold_cost_analysis.csv", index=False)
print("\nSaved full threshold sweep: dSets/threshold_cost_analysis.csv")

=== Cost at default threshold (0.5) ===
 threshold  n_flagged  false_positives  false_negatives  fp_cost  fn_cost  total_cost
       0.5         52               19              167      760 38080.15    38840.15

=== Cost-optimal threshold ===
    0.16
  512.00
  345.00
   33.00
13800.00
 7685.20
21485.20

Savings from switching to optimal threshold: ₹17354.95

Saved full threshold sweep: dSets/threshold_cost_analysis.csv


In [14]:
import pandas as pd
import numpy as np

preds = pd.read_csv("dSets/test_predictions.csv")
X_test = pd.read_csv("dSets/X_test_FINAL.csv")
preds["order_value"] = X_test["order_value"].values

y_true = preds["y_true"]
probs = preds["log_reg_prob"]
order_value = preds["order_value"]

def fn_cost(order_value):
    return 150 + 50 + 0.02 * order_value

def find_optimal_threshold(fp_cost, thresholds=np.arange(0.05, 0.95, 0.01)):
    results = []
    for t in thresholds:
        y_pred = (probs >= t).astype(int)
        fp_mask = (y_pred == 1) & (y_true == 0)
        fn_mask = (y_pred == 0) & (y_true == 1)

        total_fp_cost = fp_mask.sum() * fp_cost
        total_fn_cost = fn_cost(order_value[fn_mask]).sum()
        total_cost = total_fp_cost + total_fn_cost

        results.append({
            "threshold": round(t, 2),
            "n_flagged": y_pred.sum(),
            "total_cost": total_cost,
        })
    results_df = pd.DataFrame(results)
    best = results_df.loc[results_df["total_cost"].idxmin()]

    # cost at fixed default threshold 0.5 for comparison
    default = results_df.loc[results_df["threshold"] == 0.5]

    return best, default

print("=== Sensitivity: optimal threshold under different FP cost assumptions ===\n")
for fp_cost in [40, 80, 100, 150, 200]:
    best, default = find_optimal_threshold(fp_cost)
    savings = default["total_cost"].values[0] - best["total_cost"]
    print(f"FP_COST = ₹{fp_cost:>4}  ->  optimal threshold = {best['threshold']:.2f}, "
          f"orders flagged = {int(best['n_flagged']):>4}, "
          f"total cost = ₹{best['total_cost']:,.0f}, "
          f"savings vs default = ₹{savings:,.0f}")

=== Sensitivity: optimal threshold under different FP cost assumptions ===

FP_COST = ₹  40  ->  optimal threshold = 0.16, orders flagged =  512, total cost = ₹21,485, savings vs default = ₹17,355
FP_COST = ₹  80  ->  optimal threshold = 0.28, orders flagged =  245, total cost = ₹31,317, savings vs default = ₹8,283
FP_COST = ₹ 100  ->  optimal threshold = 0.29, orders flagged =  230, total cost = ₹33,864, savings vs default = ₹6,117
FP_COST = ₹ 150  ->  optimal threshold = 0.30, orders flagged =  216, total cost = ₹40,034, savings vs default = ₹896
FP_COST = ₹ 200  ->  optimal threshold = 0.51, orders flagged =   49, total cost = ₹41,723, savings vs default = ₹157


In [15]:
import pandas as pd
import numpy as np

X_trainpool = pd.read_csv("dSets/X_trainpool.csv")
y_trainpool = pd.read_csv("dSets/y_trainpool.csv").squeeze()

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log_reg_final = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
log_reg_final.fit(X_trainpool, y_trainpool)

# Pull out learned coefficients
coefs = log_reg_final.named_steps["model"].coef_[0]
feature_names = X_trainpool.columns

coef_df = pd.DataFrame({
    "feature": feature_names,
    "learned_coefficient": coefs
}).sort_values("learned_coefficient", key=abs, ascending=False)

print("=== Learned coefficients (sorted by magnitude) ===")
print(coef_df.to_string(index=False))

print("\n=== Compare against injected ground-truth weights (Step 1 formula) ===")
print("""
Injected weight       Feature                    Expected direction
+1.1                  payment_method_COD          positive (higher risk)
+1.8                  discount_pct                positive
+0.9                  is_first_time_buyer          positive
+0.7                  size_variant_flag            positive
-0.4                  order_value (normalized)     negative (grounded, flipped)
category: fashion +0.6, grocery -0.9               fashion positive, grocery negative
tier: tier3 +0.3, metro -0.1                        tier3 positive, metro negative
""")

=== Learned coefficients (sorted by magnitude) ===
                      feature  learned_coefficient
                  order_value            -0.436502
            size_variant_flag             0.378913
             category_grocery            -0.346062
           payment_method_COD             0.295294
                 discount_pct             0.278485
    customer_past_return_rate             0.256506
             category_fashion             0.178913
           has_return_history            -0.163787
          is_first_time_buyer             0.163787
           payment_method_UPI            -0.148524
         category_electronics            -0.144855
                category_home             0.129406
  delivery_pincode_tier_tier3             0.123018
          payment_method_card            -0.112976
  delivery_pincode_tier_metro            -0.105430
    payment_method_netbanking            -0.088543
              category_beauty             0.073788
    time_of_day_ordered_night  